In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t079'
task_file_path = f'{task_directory}/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "inventory接口流量下跌告警" (inventory interface traffic drop alert). It occurred between 2026-04-02 23:06:32 and 2026-04-02 23:09:54 (UTC+8). The entity that triggered the alert is an APM operation named "inventory::/api/v1/inventory/{productId}".

The original code execution output of IPython Kernel is also provided below for reference:

('inventory接口流量下跌告警', {'start': '2026-04-02T23:06:32.535684+08:00', 'end': '2026-04-02T23:09:54+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all unique KPIs available in the dataset
unique_kpis = metrics_data['metric'].unique()
unique_kpis

```
Out[2]:
```
The dataset contains various KPIs, including traffic-related ones such as `request_count`, `error_count`, `error_rate`, `latency`, and `slow`. These KPIs are relevant for analyzing traffic issues.

The original code execution output of IPython Kernel is also provided below for reference:

['node_ready_status' 'node_not_ready_status' 'node_disk_usage_rate'
 'node_memory_usage_rate' 'node_cpu_usage_rate' 'node_pod_running_count'
 'deployment_desired_replicas' 'deployment_available_replicas'
 'deployment_ready_replicas' 'deployment_availability_rate'
 'deployment_memory_usage_total' 'deployment_cpu_usage_total'
 'deployment_cpu_usage_vs_requests' 'deployment_cpu_usage_vs_limits'
 'cluster_node_count' 'cluster_failed_node_count' 'cluster_number_of_pod'
 'cluster_number_of_not_ready_pods' 'apiserver_requests_per_second'
 'apiserver_non_2xx_write_request_qps' 'apiserver_concurrent_connections'
 'apiserver_non_2xx_read_request_qps' 'namespace_number_of_not_ready_pods'
 'namespace_number_of_pods' 'error_count' 'slow_count' 'request_count'
 'avg_request_latency_seconds' 'arms_jvm_gc_total'
 'arms_jvm_gc_seconds_total' 'arms_jvm_gc_delta'
 'arms_jvm_gc_seconds_delta' 'arms_jvm_mem_used_bytes'
 'arms_jvm_mem_init_bytes' 'arms_jvm_mem_committed_bytes'
 'arms_jvm_mem_max_bytes' 'arms_jvm_threads_count'
 'arms_thread_pool_active_thread_count'
 'arms_thread_pool_current_thread_count'
 'arms_thread_pool_max_thread_count' 'arms_thread_pool_max_pool_size'
 'arms_thread_pool_core_pool_size' 'arms_thread_pool_scheduled_task_count'
 'arms_thread_pool_completed_task_count'
 'arms_thread_pool_rejected_task_count' 'arms_thread_pool_queue_size'
 'arms_grouped_thread_count' 'arms_grouped_thread_cpu_time'
 'arms_grouped_thread_runnable_count' 'arms_grouped_thread_waiting_count'
 'arms_grouped_thread_timed_waiting_count' 'cpu' 'latency' 'mem'
 'workload' 'slow' 'error' 'error_rate']```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Convert the alert window timestamps to microseconds for filtering
timezone = pytz.timezone('Asia/Shanghai')
start_time = int(datetime.strptime(alert_window['start'], "%Y-%m-%dT%H:%M:%S.%f%z").timestamp() * 1_000_000)
end_time = int(datetime.strptime(alert_window['end'], "%Y-%m-%dT%H:%M:%S%z").timestamp() * 1_000_000)

# Filter metrics data for the relevant KPIs
relevant_kpis = ['request_count', 'error_count', 'error_rate', 'latency', 'slow']
filtered_metrics = metrics_data[metrics_data['metric'].isin(relevant_kpis)]

# Ensure 'time' and 'value' columns are numeric
filtered_metrics['time'] = pd.to_numeric(filtered_metrics['time'], errors='coerce')
filtered_metrics['value'] = pd.to_numeric(filtered_metrics['value'], errors='coerce')

# Calculate global P95 thresholds for each KPI across all components
global_p95_thresholds = filtered_metrics.groupby('metric')['value'].quantile(0.95)

# Filter data within the alert time window
time_window_metrics = filtered_metrics[(filtered_metrics['time'] >= start_time) & (filtered_metrics['time'] <= end_time)]

global_p95_thresholds, time_window_metrics

```
Out[3]:
```
The global P95 thresholds for the relevant KPIs across all components are as follows:
- `error_count`: 33
- `error_rate`: 0.072040
- `latency`: 0.248735 seconds
- `request_count`: 13,037.25
- `slow`: 2,222

Within the alert time window (2026-04-02 23:06:32 to 2026-04-02 23:09:54 UTC+8), 584 rows of metrics data were filtered for these KPIs across various components. This data will help identify components exceeding the thresholds during the incident.

The original code execution output of IPython Kernel is also provided below for reference:

(metric
error_count         33.000000
error_rate           0.072040
latency              0.248735
request_count    13037.250000
slow              2222.000000
Name: value, dtype: float64,                    time domain     entity_set                         entity_id                         entity_name       metric      value                            metric_set_id service
53338  1775142482000000    apm  apm.operation  fb364e80b42f16618b6de13f3579b6ab  frontend::GET /api/recommendations  error_count   7.000000  apm@metric_set@apm.metric.apm.operation    None
53339  1775142512000000    apm  apm.operation  fb364e80b42f16618b6de13f3579b6ab  frontend::GET /api/recommendations  error_count  15.000000  apm@metric_set@apm.metric.apm.operation    None
53340  1775142542000000    apm  apm.operation  fb364e80b42f16618b6de13f3579b6ab  frontend::GET /api/recommendations  error_count  20.000000  apm@metric_set@apm.metric.apm.operation    None
53341  1775142572000000    apm  apm.operation  fb364e80b42f16618b6de13f3579b6ab  frontend::GET /api/recommendations  error_count  34.000000  apm@metric_set@apm.metric.apm.operation    None
53356  1775142482000000    apm  apm.operation  d1c2b3981457bf9b5c5beefbbf246ebc             frontend-proxy::ingress  error_count   7.000000  apm@metric_set@apm.metric.apm.operation    None
...                 ...    ...            ...                               ...                                 ...          ...        ...                                      ...     ...
83170  1775142572000000    apm   apm.instance  009ec66a350bc3e5210aea3dc6e6bdfd           frontend-6df7d557f5-sgfcb   error_rate   0.002515   apm@metric_set@apm.metric.apm.instance    None
83183  1775142482000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609     frontend-proxy-67d64dc4f8-4ptg6   error_rate   0.000511   apm@metric_set@apm.metric.apm.instance    None
83184  1775142512000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609     frontend-proxy-67d64dc4f8-4ptg6   error_rate   0.000644   apm@metric_set@apm.metric.apm.instance    None
83185  1775142542000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609     frontend-proxy-67d64dc4f8-4ptg6   error_rate   0.000804   apm@metric_set@apm.metric.apm.instance    None
83186  1775142572000000    apm   apm.instance  eab531e9b188fe065edbebdbf839c609     frontend-proxy-67d64dc4f8-4ptg6   error_rate   0.001651   apm@metric_set@apm.metric.apm.instance    None

[584 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Identify components and KPIs with consecutive anomalies exceeding the global P95 thresholds
anomalous_components = []

# Iterate through each KPI and check for anomalies
for kpi, threshold in global_p95_thresholds.items():
    # Filter data for the current KPI
    kpi_data = time_window_metrics[time_window_metrics['metric'] == kpi]
    
    # Identify rows where the value exceeds the global P95 threshold
    kpi_data['is_anomalous'] = kpi_data['value'] > threshold
    
    # Group by entity and check for consecutive anomalies
    kpi_data['consecutive_anomalies'] = kpi_data.groupby('entity_id')['is_anomalous'].cumsum()
    anomalous_entities = kpi_data[kpi_data['is_anomalous']].groupby('entity_id').size()
    
    # Add to the list of anomalous components if anomalies are found
    if not anomalous_entities.empty:
        anomalous_components.append((kpi, anomalous_entities))

# Display the anomalous components and their KPIs
anomalous_components

```
Out[4]:
```
During the alert time window, the following components exhibited consecutive anomalies exceeding the global P95 thresholds:

- For `error_count`: anomalies were detected in three components with entity IDs `28887014538b3aeed572d8e4810f6d88`, `d1c2b3981457bf9b5c5beefbbf246ebc`, and `fb364e80b42f16618b6de13f3579b6ab`.
- For `error_rate`: anomalies were detected in one component with entity ID `fb364e80b42f16618b6de13f3579b6ab`.
- For `request_count`: anomalies were detected in three components with entity IDs `82ae8fb5979d5759841affc9d937a0d4` (2 anomalies), `beecfe3febef4ed2425bac5626962ed9` (3 anomalies), and `d1c2b3981457bf9b5c5beefbbf246ebc` (4 anomalies).

These components and KPIs are likely contributing to the observed fault.

The original code execution output of IPython Kernel is also provided below for reference:

[('error_count', entity_id
28887014538b3aeed572d8e4810f6d88    1
d1c2b3981457bf9b5c5beefbbf246ebc    1
fb364e80b42f16618b6de13f3579b6ab    1
dtype: int64), ('error_rate', entity_id
fb364e80b42f16618b6de13f3579b6ab    1
dtype: int64), ('request_count', entity_id
82ae8fb5979d5759841affc9d937a0d4    2
beecfe3febef4ed2425bac5626962ed9    3
d1c2b3981457bf9b5c5beefbbf246ebc    4
dtype: int64)]```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime' and 'endTime' columns are numeric
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces within the alert time window
filtered_traces = traces_data[
    (traces_data['startTime'] >= start_time * 1_000) & 
    (traces_data['endTime'] <= end_time * 1_000)
]

# Focus on traces involving the identified faulty components
faulty_components = [
    '28887014538b3aeed572d8e4810f6d88', 
    'd1c2b3981457bf9b5c5beefbbf246ebc', 
    'fb364e80b42f16618b6de13f3579b6ab', 
    '82ae8fb5979d5759841affc9d937a0d4', 
    'beecfe3febef4ed2425bac5626962ed9'
]
component_traces = filtered_traces[filtered_traces['traceId'].isin(faulty_components)]

# Determine the most downstream faulty service in the call chain
downstream_faults = component_traces.sort_values(by='startTime', ascending=False)

downstream_faults

```
Out[5]:
```
No traces were found within the alert time window (2026-04-02 23:06:32 to 2026-04-02 23:09:54 UTC+8) involving the identified faulty components. This suggests that trace data does not provide additional insights into the most downstream faulty service in the call chain for this incident.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, links, ip]
Index: []

[0 rows x 21 columns]```
```

In [ ]:
# In[6]:

# Revise the code to address the KeyError issue
# Inspect the columns in the logs dataset to ensure correct filtering
logs_columns = logs_data.columns

# Display the columns to identify the correct ones for filtering
logs_columns

```
Out[6]:
```
The `logs.parquet` dataset contains columns such as `content` (log message), `_time_` (timestamp), `_namespace_`, `_pod_name_`, and others. These columns can be used to filter logs within the alert time window and focus on the identified faulty components.

The original code execution output of IPython Kernel is also provided below for reference:

Index(['content', '_time_', '_source_', '_namespace_', '_pod_uid_', '_container_ip_', '_image_name_', '_container_name_', '_pod_name_', '__topic__', '__tag__:__pack_id__', '__tag__:__hostname__', '__tag__:_node_name_', '__tag__:_node_ip_', '__tag__:_cluster_id_'], dtype='object')```
```

In [ ]:
# In[7]:

# Ensure '_time_' column is numeric for filtering
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_data[(logs_data['_time_'] >= start_time) & (logs_data['_time_'] <= end_time)]

# Search for logs related to the identified faulty components
faulty_components_logs = filtered_logs[
    filtered_logs['_pod_uid_'].isin([
        '28887014538b3aeed572d8e4810f6d88', 
        'd1c2b3981457bf9b5c5beefbbf246ebc', 
        'fb364e80b42f16618b6de13f3579b6ab', 
        '82ae8fb5979d5759841affc9d937a0d4', 
        'beecfe3febef4ed2425bac5626962ed9'
    ])
]

# Search for error messages or patterns in the 'content' column
error_logs = faulty_components_logs[faulty_components_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[7]:
```
No logs containing error messages or patterns were found within the alert time window (2026-04-02 23:06:32 to 2026-04-02 23:09:54 UTC+8) for the identified faulty components. This suggests that the logs do not provide additional insights into the root cause of the issue.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```